<a href="https://colab.research.google.com/github/mizinco/sam2-recognition-limit-observer/blob/main/sam2_video_predictor_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM2 Video Predictor — ノートブック

Week 1 の最小 E2E 貫通用。Colab T4 GPU で動作することを前提に書かれています。

**目的**: 任意の車載クリップに対しクリック1点で SAM2 video predictor を動かし、アノテーション付き MP4 を出力する。

## 進め方

1. ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = T4 GPU
2. 上から順にセルを実行
3. 「自前クリップ差し替え」セクションで自分の動画に切り替え

## 1. 環境チェック

In [1]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Tue May 19 07:07:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. SAM2 のインストール

公式リポジトリを clone し、editable インストール。

In [2]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -e . -q
!pip install opencv-python matplotlib -q

/content
Cloning into 'sam2'...
remote: Enumerating objects: 1096, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 1096 (delta 1), reused 0 (delta 0), pack-reused 1094 (from 2)
Receiving objects: 100% (1096/1096), 134.84 MiB | 14.99 MiB/s, done.
Resolving deltas: 100% (376/376), done.
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.6 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done


## 3. チェックポイントのダウンロード

SAM2.1 Hiera-Large を使用。Tiny/Small/Base+ も同梱スクリプトで取得可能。

In [3]:
%cd /content/sam2/checkpoints
!./download_ckpts.sh
!ls -lh /content/sam2/checkpoints/

/content/sam2/checkpoints
--2026-05-19 07:10:15--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.4, 65.9.168.62, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156008466 (149M) [application/vnd.snesdev-page-table]
Saving to: ‘sam2.1_hiera_tiny.pt’

sam2.1_hiera_tiny.p 100%[===================>] 148.78M   202MB/s    in 0.7s    

2026-05-19 07:10:16 (202 MB/s) - ‘sam2.1_hiera_tiny.pt’ saved [156008466/156008466]

--2026-05-19 07:10:16--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.4, 65.9.168.62, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 184416285 (1

## 4. インポートと device 設定

In [4]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import cv2

sys.path.insert(0, '/content/sam2')
from sam2.build_sam import build_sam2_video_predictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.autocast(device_type=device.type, dtype=torch.bfloat16).__enter__()
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print('using device:', device)

using device: cuda


## 5. predictor のロード

In [5]:
sam2_checkpoint = '/content/sam2/checkpoints/sam2.1_hiera_large.pt'
model_cfg = 'configs/sam2.1/sam2.1_hiera_l.yaml'
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
print('predictor ready')

predictor ready


## 6. 動画 → JPEG フレーム展開

SAM2 video predictor はフレームディレクトリを入力に取るため、先に展開する。

**初回確認用**: 公式リポジトリ同梱のサンプル動画を使用。  
**自前クリップ差し替え**: 下のセルで `VIDEO_PATH` を書き換える。

In [ ]:
%%bash
# まず既存ファイルを完全削除し、削除を目視確認
rm -f /content/2011_09_26_drive_0001_sync.zip
echo "=== 削除確認（何も出なければOK）==="
ls /content/2011_09_26_drive_0001_sync.zip 2>&1

# 念のため過去の解凍も消す
rm -rf /content/kitti
rm -rf /content/2011_09_26

# wget で1回だけ取得。-O で必ず指定パスに上書き
echo ""
echo "=== ダウンロード開始 ==="
wget --progress=dot:giga \
  -O /content/2011_09_26_drive_0001_sync.zip \
  "https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0001/2011_09_26_drive_0001_sync.zip"

# ダウンロード後の検証（3段階）
echo ""
echo "=== サイズ確認（数百MB前後が期待値）==="
ls -lh /content/2011_09_26_drive_0001_sync.zip

echo ""
echo "=== ZIP 整合性テスト ==="
unzip -t /content/2011_09_26_drive_0001_sync.zip > /tmp/ziptest.log 2>&1
RESULT=$?
tail -3 /tmp/ziptest.log
echo ""
if [ $RESULT -eq 0 ]; then
  echo "OK: 解凍テスト成功。次のセルで実際の解凍に進めます"
else
  echo "NG: まだ壊れています。Colab を一度ランタイムごと再起動してから再実行してください"
fi

In [ ]:
%%bash
# 解凍
echo "=== 解凍中 ==="
unzip -q /content/2011_09_26_drive_0001_sync.zip -d /content/kitti
echo "解凍完了"

In [ ]:
%%bash
# image_02 (カラー左カメラ) の確認
FRAME_DIR=/content/kitti/2011_09_26/2011_09_26_drive_0001_sync/image_02/data
echo ""
echo "=== フレーム確認 ==="
echo "総数: $(ls $FRAME_DIR | wc -l)"
echo "先頭: $(ls $FRAME_DIR | head -3)"
echo "末尾: $(ls $FRAME_DIR | tail -3)"

In [ ]:
%%bash
# MP4 化
echo ""
echo "=== MP4 化中 ==="
ffmpeg -y -framerate 10 -i /content/kitti/2011_09_26/2011_09_26_drive_0001_sync/image_02/data/%010d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -pix_fmt yuv420p /content/kitti_drive_0001.mp4

In [ ]:
%%bash
echo ""
echo "=== 結果 ==="

In [ ]:
!ls -lh /content/kitti_drive_0001.mp4
!ffprobe -v error -show_entries format=duration -show_entries stream=width,height,nb_frames -of default=noprint_wrappers=1 /content/kitti_drive_0001.mp4

In [ ]:
import os

USE_OFFICIAL_SAMPLE = False
VIDEO_PATH = '/content/kitti_drive_0001.mp4'
frame_dir = '/content/frames'
os.makedirs(frame_dir, exist_ok=True)

!rm -f /content/frames/*.jpg
!ffmpeg -y -i {VIDEO_PATH} -q:v 2 -start_number 0 {frame_dir}/%05d.jpg

frame_names = sorted([f for f in os.listdir(frame_dir) if f.lower().endswith(('.jpg', '.jpeg'))])
print(f'total frames: {len(frame_names)}')

In [ ]:
# === 設定: ここを書き換えると自前クリップに差し替えられます ===
# 公式サンプル(寝室) は notebooks/videos/bedroom にフレームが既にある
USE_OFFICIAL_SAMPLE = False

if USE_OFFICIAL_SAMPLE:
    frame_dir = '/content/sam2/notebooks/videos/bedroom'
    print('using official sample frames at:', frame_dir)


## 7. 推論状態の初期化

In [ ]:
inference_state = predictor.init_state(video_path=frame_dir)
print('inference state initialized')

## 8. 追跡開始フレームを表示しクリック点を決める

最初のフレームを表示し、目で見て追跡対象の (x, y) を決める。

In [ ]:
ann_frame_idx = 0
img = Image.open(os.path.join(frame_dir, frame_names[ann_frame_idx]))
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title(f'frame {ann_frame_idx} — 追跡したい物体の中心 (x, y) を読み取る')
plt.axis('on')
plt.show()
print('画像サイズ:', img.size)

###cyclist_001


In [ ]:
click_xy = (770, 175)  # cyclist_001
CASE_NAME = 'cyclist_001'

###tram_001

In [ ]:
click_xy = (725, 200)  #  tram_001
CASE_NAME = 'tram_001'

###parked_car_001

In [ ]:
click_xy = (725, 200)  #  parked_car_001
CASE_NAME = 'parked_car_001'

###マスク可視化

In [ ]:
points = np.array([[click_xy[0], click_xy[1]]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)  # 1 = positive (追跡対象)

frame_idx, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=1,
    points=points,
    labels=labels,
)

# クリック点と最初のマスクを可視化
mask = (out_mask_logits[0] > 0.0).cpu().numpy()
if mask.ndim == 3:
    mask = mask[0]
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.imshow(mask, alpha=0.5, cmap='Reds')
plt.scatter([click_xy[0]], [click_xy[1]], c='lime', s=20, marker='*', edgecolors='black')
plt.title('frame 0 — click + initial mask')
plt.axis('off')
plt.show()

## 9. 全フレームへ伝播

In [ ]:
video_segments = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
print('propagation done. frames covered:', len(video_segments))

## 10. 結果を MP4 として書き出す

In [ ]:
# CASE_NAME = 'cyclist_001'  # 試行ごとに変える: parked_car_001, tram_001, cyclist_001 など
OUTPUT_DIR = f'/content/output/{CASE_NAME}'
OVERLAY_DIR = os.path.join(OUTPUT_DIR, 'overlay_frames')
os.makedirs(OVERLAY_DIR, exist_ok=True)

In [ ]:
for idx, fname in enumerate(frame_names):
    img_bgr = cv2.imread(os.path.join(frame_dir, fname))
    if idx in video_segments:
        for oid, m in video_segments[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            color = np.array([0, 0, 255], dtype=np.uint8)  # BGR red
            overlay = img_bgr.copy()
            overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 + color * 0.5).astype(np.uint8)
            img_bgr = overlay
    cv2.imwrite(os.path.join(OVERLAY_DIR, f'{idx:05d}.jpg'), img_bgr)

out_mp4 = os.path.join(OUTPUT_DIR, f'{CASE_NAME}.mp4')
!ffmpeg -y -framerate 24 -i "{OVERLAY_DIR}/%05d.jpg" -c:v libx264 -pix_fmt yuv420p "{out_mp4}"
print('saved:', out_mp4)

In [ ]:
# Colab 上で確認
from IPython.display import Video
Video(out_mp4, embed=True, width=720)

In [ ]:
# ダウンロードしたいフォルダを zip 圧縮する
!zip -r /content/output/{CASE_NAME}.zip {OUTPUT_DIR}

# 圧縮した zip ファイルをダウンロードする
from google.colab import files
files.download(f"/content/output/{CASE_NAME}.zip")

In [ ]:
# === マスクデータを .npz として保存（指標計算用）===
import numpy as np

#CASE_NAME = 'cyclist_001'  # 試行ごとに変える: parked_car_001, tram_001, cyclist_001 など

# 画像サイズ取得
img0 = Image.open(os.path.join(frame_dir, frame_names[0]))
W, H = img0.size
print(f'image size: W={W}, H={H}')

# 全フレームのマスクを 1 つの 3D 配列に詰める
masks_arr = np.zeros((len(frame_names), H, W), dtype=bool)
for idx in range(len(frame_names)):
    if idx in video_segments:
        for oid, m in video_segments[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            masks_arr[idx] = mask_arr.astype(bool)

# 保存
save_dir = f'/content/output/{CASE_NAME}'
os.makedirs(save_dir, exist_ok=True)
out_path = f'{save_dir}/masks.npz'
np.savez_compressed(out_path,
                    masks=masks_arr,
                    click_xy=np.array(click_xy, dtype=np.float32),
                    case_name=np.array([CASE_NAME]),
                    H=np.array([H]),
                    W=np.array([W]),
                    fps=np.array([10]))

print(f'masks shape: {masks_arr.shape}')
print(f'non-zero frames: {(masks_arr.sum(axis=(1,2)) > 0).sum()} / {len(frame_names)}')
print(f'saved to: {out_path}')
print(f'compressed file size: {os.path.getsize(out_path)/1e6:.2f} MB')

#Week 2 指標計算 & 可視化

In [ ]:
# === Week 2 指標計算 & 可視化（指標4: 形状IoU を追加した更新版）===
import os, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

# --- 指標関数群 ---

def mask_area_change_rate(masks):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    rate = np.zeros(len(areas), dtype=np.float32)
    for t in range(1, len(areas)):
        prev, curr = areas[t-1], areas[t]
        if prev == 0:
            rate[t] = 0.0 if curr == 0 else float('inf')
        else:
            rate[t] = (curr - prev) / prev
    return rate

def connected_components_count(masks, min_size=50):
    counts = np.zeros(len(masks), dtype=np.int32)
    for t in range(len(masks)):
        if masks[t].sum() == 0:
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0: continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        counts[t] = int((sizes >= min_size).sum())
    return counts

def largest_component_ratio(masks):
    ratios = np.ones(len(masks), dtype=np.float32)
    for t in range(len(masks)):
        total = masks[t].sum()
        if total == 0: continue
        labeled, n = ndimage.label(masks[t])
        if n == 0: continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        ratios[t] = float(sizes.max() / total)
    return ratios

def mask_disappearance(masks, threshold_abs=50, threshold_rel=0.01, ref_frames=5):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    ref = areas[:max(ref_frames, 1)]
    ref_area = float(ref.mean()) if len(ref) > 0 else 1.0
    threshold = max(float(threshold_abs), ref_area * threshold_rel)
    is_dis = areas < threshold
    cons = np.zeros(len(areas), dtype=np.int32)
    for t in range(len(areas)):
        cons[t] = (cons[t-1] if t > 0 else 0) + 1 if is_dis[t] else 0
    return {'areas': areas, 'is_disappeared': is_dis, 'consecutive': cons,
            'threshold': threshold, 'reference_area': ref_area}

def shape_iou_temporal(masks):
    """重心揃え後の連続フレーム IoU。位置の変化を打ち消して形状の変化だけを評価"""
    T = len(masks)
    iou = np.ones(T, dtype=np.float32)
    if T == 0: return iou
    H, W = masks[0].shape
    def centroid(m):
        ys, xs = np.where(m)
        return (float(ys.mean()), float(xs.mean())) if len(ys) > 0 else None
    for t in range(1, T):
        mp, mc = masks[t-1], masks[t]
        if mp.sum() == 0 or mc.sum() == 0:
            iou[t] = 1.0
            continue
        cp, cc = centroid(mp), centroid(mc)
        dy = int(round(cp[0] - cc[0]))
        dx = int(round(cp[1] - cc[1]))
        shifted = np.zeros_like(mc)
        y0, y1 = max(0, dy), min(H, H + dy)
        x0, x1 = max(0, dx), min(W, W + dx)
        if y0 < y1 and x0 < x1:
            shifted[y0:y1, x0:x1] = mc[y0-dy:y1-dy, x0-dx:x1-dx]
        inter = np.logical_and(mp, shifted).sum()
        union = np.logical_or(mp, shifted).sum()
        iou[t] = float(inter / union) if union > 0 else 1.0
    return iou

def compute_all_metrics(masks, min_component_size=50):
    area_change = mask_area_change_rate(masks)
    components = connected_components_count(masks, min_size=min_component_size)
    largest = largest_component_ratio(masks)
    dis = mask_disappearance(masks)
    siou = shape_iou_temporal(masks)
    return {
        'frame_count': int(len(masks)),
        'areas': dis['areas'].astype(float).tolist(),
        'area_change_rate': area_change.tolist(),
        'connected_components': components.tolist(),
        'largest_component_ratio': largest.tolist(),
        'shape_iou_temporal': siou.tolist(),
        'is_disappeared': dis['is_disappeared'].tolist(),
        'consecutive_disappearance': dis['consecutive'].tolist(),
        'reference_area': float(dis['reference_area']),
        'disappearance_threshold': float(dis['threshold']),
        'min_component_size': int(min_component_size),
    }

# --- 解析と可視化（5パネル版）---

plt.rcParams['font.family'] = 'DejaVu Sans'

def analyze_one(npz_path, output_dir):
    data = np.load(npz_path, allow_pickle=True)
    masks = data['masks']
    case_name = (str(data['case_name'].item() if 'case_name' in data and data['case_name'].ndim == 0
                     else data['case_name'][0]) if 'case_name' in data
                 else Path(npz_path).parent.name)
    click_xy = data['click_xy'].tolist() if 'click_xy' in data else None
    print(f'[{case_name}] shape={masks.shape}')

    metrics = compute_all_metrics(masks)
    metrics['case_name'] = case_name
    metrics['click_xy'] = click_xy

    os.makedirs(output_dir, exist_ok=True)
    with open(f'{output_dir}/metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    fig, ax = plt.subplots(5, 1, figsize=(12, 13), sharex=True)
    frames = np.arange(len(masks))

    ax[0].plot(frames, metrics['areas'], color='steelblue', linewidth=1.5)
    ax[0].axhline(metrics['disappearance_threshold'], color='red', linestyle='--',
                  label=f"Disappearance threshold ({metrics['disappearance_threshold']:.0f}px)")
    ax[0].set_ylabel('Mask area (px)')
    ax[0].set_title(f'{case_name} - mask area over time')
    ax[0].legend(); ax[0].grid(alpha=0.3)

    rates = np.array(metrics['area_change_rate'])
    ax[1].plot(frames, np.clip(rates, -2, 2), color='darkorange', linewidth=1.5)
    ax[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='±50%')
    ax[1].axhline(-0.5, color='red', linestyle='--', alpha=0.5)
    ax[1].set_ylabel('Area change rate')
    ax[1].set_title('Indicator 1: Mask area change rate')
    ax[1].legend(); ax[1].grid(alpha=0.3)

    ax[2].plot(frames, metrics['connected_components'], color='seagreen',
               linewidth=1.5, marker='o', markersize=3)
    ax[2].set_ylabel('Component count')
    ax[2].set_title(f"Indicator 2: Connected components (>= {metrics['min_component_size']}px)")
    ax[2].grid(alpha=0.3)

    siou = np.array(metrics['shape_iou_temporal'])
    ax[3].plot(frames, siou, color='purple', linewidth=1.5)
    ax[3].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='IoU 0.8')
    ax[3].set_ylim(0, 1.05)
    ax[3].set_ylabel('Shape IoU')
    ax[3].set_title('Indicator 4: Shape stability (centroid-aligned IoU vs previous frame)')
    ax[3].legend(); ax[3].grid(alpha=0.3)

    is_dis = np.array(metrics['is_disappeared'])
    cons = np.array(metrics['consecutive_disappearance'])
    ax[4].fill_between(frames, 0, is_dis.astype(int), alpha=0.4, color='crimson',
                       step='post', label='Disappeared')
    if cons.max() > 0:
        ax[4].plot(frames, cons / max(cons.max(), 1), color='crimson', linewidth=1.5,
                   label='Consecutive (normalized)')
    ax[4].set_ylim(-0.05, 1.1)
    ax[4].set_ylabel('Disappearance')
    ax[4].set_title(f"Indicator 3: Mask disappearance (max consecutive: {int(cons.max())} frames)")
    ax[4].set_xlabel('Frame')
    ax[4].legend(); ax[4].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/metrics.png', dpi=120, bbox_inches='tight')
    plt.close()

    print(f'  -> {output_dir}/metrics.json, metrics.png')
    print(f'  - frames with shape IoU < 0.8: {int((siou < 0.8).sum())}')
    print(f'  - min shape IoU: {float(siou.min()):.3f} at frame {int(siou.argmin())}')
    print(f'  - mean shape IoU (frames 1+): {float(siou[1:].mean()):.3f}')
    print(f'  - disappeared frames: {int(is_dis.sum())} / {len(masks)}')
    print(f'  - max components: {int(max(metrics["connected_components"]))}')
    print(f'  - abrupt area change frames (|rate|>0.5): {int((np.abs(rates) > 0.5).sum())}')
    return metrics

cases = ['parked_car_001', 'tram_001', 'cyclist_001']
all_metrics = {}
for case in cases:
    print('=' * 60)
    npz_path = f'/content/output/{case}/masks.npz'
    if not os.path.exists(npz_path):
        print(f'SKIP: {npz_path} not found')
        continue
    all_metrics[case] = analyze_one(npz_path, f'/content/output/{case}')
print()
print('=' * 60)
print(f'Done. {len(all_metrics)} cases analyzed.')

In [ ]:
!cd /content/output && \
  zip -r week2_results_v2.zip \
    parked_car_001/masks.npz parked_car_001/metrics.json parked_car_001/metrics.png \
    tram_001/masks.npz tram_001/metrics.json tram_001/metrics.png \
    cyclist_001/masks.npz cyclist_001/metrics.json cyclist_001/metrics.png && \
  ls -lh week2_results_v2.zip

##Task 2: 異条件クリップ 2本を追加
###Colab でダウンロード

In [ ]:
# クリップA: drive_0013 (日陰多めシーン)
!wget -q --show-progress \
  https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0013/2011_09_26_drive_0013_sync.zip \
  -O /content/2011_09_26_drive_0013_sync.zip

# クリップB: drive_0017 (電柱越し遮蔽シーン)
!wget -q --show-progress \
  https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0017/2011_09_26_drive_0017_sync.zip \
  -O /content/2011_09_26_drive_0017_sync.zip

# 整合性チェック
!ls -lh /content/2011_09_26_drive_001*_sync.zip
!unzip -t /content/2011_09_26_drive_0013_sync.zip > /tmp/t13.log 2>&1 && tail -2 /tmp/t13.log
!unzip -t /content/2011_09_26_drive_0017_sync.zip > /tmp/t17.log 2>&1 && tail -2 /tmp/t17.log

###解凍 + MP4 化

In [ ]:
# 過去の解凍データを掃除
!rm -rf /content/kitti/2011_09_26/2011_09_26_drive_0013_sync
!rm -rf /content/kitti/2011_09_26/2011_09_26_drive_0017_sync
!rm -f /content/kitti_drive_0013.mp4 /content/kitti_drive_0017.mp4

# 解凍
!unzip -q /content/2011_09_26_drive_0013_sync.zip -d /content/kitti
!unzip -q /content/2011_09_26_drive_0017_sync.zip -d /content/kitti

# フレーム数確認
!echo "drive_0013 frames: $(ls /content/kitti/2011_09_26/2011_09_26_drive_0013_sync/image_02/data/ | wc -l)"
!echo "drive_0017 frames: $(ls /content/kitti/2011_09_26/2011_09_26_drive_0017_sync/image_02/data/ | wc -l)"

# MP4 化（パディング付き、奇数解像度対策）
!ffmpeg -y -framerate 10 \
  -i /content/kitti/2011_09_26/2011_09_26_drive_0013_sync/image_02/data/%010d.png \
  -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
  -c:v libx264 -pix_fmt yuv420p \
  /content/kitti_drive_0013.mp4

!ffmpeg -y -framerate 10 \
  -i /content/kitti/2011_09_26/2011_09_26_drive_0017_sync/image_02/data/%010d.png \
  -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
  -c:v libx264 -pix_fmt yuv420p \
  /content/kitti_drive_0017.mp4

!ls -lh /content/kitti_drive_0013.mp4 /content/kitti_drive_0017.mp4

##drive_0013（A: 日陰）
###新セル 1: drive_0013 フレーム展開

In [ ]:
# === Week 2 追加: drive_0013 (日陰多めシーン) ===
import os
VIDEO_PATH_D13 = '/content/kitti_drive_0013.mp4'
frame_dir_d13 = '/content/frames_d13'
os.makedirs(frame_dir_d13, exist_ok=True)
!rm -f /content/frames_d13/*.jpg
!ffmpeg -y -i {VIDEO_PATH_D13} -q:v 2 -start_number 0 {frame_dir_d13}/%05d.jpg -loglevel error

frame_names_d13 = sorted([f for f in os.listdir(frame_dir_d13) if f.lower().endswith(('.jpg', '.jpeg'))])
print(f'drive_0013 frames extracted: {len(frame_names_d13)}')

###新セル 2: drive_0013 用の inference_state 初期化

In [ ]:
# state リセット
try:
    predictor.reset_state(inference_state_d13)
except (NameError, AttributeError):
    pass
inference_state_d13 = predictor.init_state(video_path=frame_dir_d13)
print('drive_0013 state initialized')

###新セル 3: frame 0 を表示

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

img_d13 = Image.open(os.path.join(frame_dir_d13, frame_names_d13[0]))
plt.figure(figsize=(14, 5))
plt.imshow(img_d13)
plt.title(f'drive_0013 - frame 0 (size: {img_d13.size})')
plt.axis('on')
plt.grid(True, alpha=0.3)
plt.show()
print('image size:', img_d13.size)

###新セル 4: クリック + 初期マスク確認

In [ ]:
click_xy_d13 = (1000, 360)  # 黒い車のドアパネル下部

points = np.array([[click_xy_d13[0], click_xy_d13[1]]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)

frame_idx, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state_d13,
    frame_idx=0,
    obj_id=1,
    points=points,
    labels=labels,
)

mask = (out_mask_logits[0] > 0.0).cpu().numpy()
if mask.ndim == 3:
    mask = mask[0]
plt.figure(figsize=(14, 5))
plt.imshow(img_d13)
plt.imshow(mask, alpha=0.5, cmap='Reds')
plt.scatter([click_xy_d13[0]], [click_xy_d13[1]], c='lime', s=120, marker='*', edgecolors='black')
plt.title(f'drive_0013 frame 0 - click {click_xy_d13} + initial mask')
plt.axis('off')
plt.show()

###新セル 5: 全フレーム伝播

In [ ]:
video_segments_d13 = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state_d13):
    video_segments_d13[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
print('drive_0013 propagation done. frames covered:', len(video_segments_d13))

###新セル 6: マスク保存

In [ ]:
CASE_NAME_D13 = 'shadow_001'

img0 = Image.open(os.path.join(frame_dir_d13, frame_names_d13[0]))
W, H = img0.size
masks_arr_d13 = np.zeros((len(frame_names_d13), H, W), dtype=bool)
for idx in range(len(frame_names_d13)):
    if idx in video_segments_d13:
        for oid, m in video_segments_d13[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            masks_arr_d13[idx] = mask_arr.astype(bool)

save_dir = f'/content/output/{CASE_NAME_D13}'
os.makedirs(save_dir, exist_ok=True)
out_path = f'{save_dir}/masks.npz'
np.savez_compressed(out_path,
                    masks=masks_arr_d13,
                    click_xy=np.array(click_xy_d13, dtype=np.float32),
                    case_name=np.array([CASE_NAME_D13]),
                    H=np.array([H]),
                    W=np.array([W]),
                    fps=np.array([10]),
                    source_drive=np.array(['2011_09_26_drive_0013']),
                    condition_tag=np.array(['shadow_alternating + same_class_overtake']))

print(f'masks shape: {masks_arr_d13.shape}')
print(f'non-zero frames: {(masks_arr_d13.sum(axis=(1,2)) > 0).sum()} / {len(frame_names_d13)}')
print(f'saved to: {out_path}')
print(f'compressed file size: {os.path.getsize(out_path)/1e6:.2f} MB')

###新セル 7: オーバーレイ MP4 を出力（後で見返すため）

In [ ]:
import cv2
OVERLAY_DIR = f'/content/output/{CASE_NAME_D13}/overlay_frames'
os.makedirs(OVERLAY_DIR, exist_ok=True)

for idx, fname in enumerate(frame_names_d13):
    img_bgr = cv2.imread(os.path.join(frame_dir_d13, fname))
    if idx in video_segments_d13:
        for oid, m in video_segments_d13[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            color = np.array([0, 0, 255], dtype=np.uint8)
            overlay = img_bgr.copy()
            overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 + color * 0.5).astype(np.uint8)
            img_bgr = overlay
    cv2.imwrite(os.path.join(OVERLAY_DIR, f'{idx:05d}.jpg'), img_bgr)

out_mp4 = f'/content/output/{CASE_NAME_D13}/{CASE_NAME_D13}.mp4'
!ffmpeg -y -framerate 10 -i "{OVERLAY_DIR}/%05d.jpg" -c:v libx264 -pix_fmt yuv420p "{out_mp4}" -loglevel error
print('saved:', out_mp4)

In [ ]:
# Colab 上で確認
from IPython.display import Video
Video(out_mp4, embed=True, width=720)

In [ ]:
!ls -lh /content/output/shadow_001/

In [ ]:
# zip 圧縮
!zip -r /content/output/{CASE_NAME_D13}.zip /content/output/{CASE_NAME_D13}

# ダウンロード
from google.colab import files
files.download(f"/content/output/{CASE_NAME_D13}.zip")

###新セル shadow_002-A: 車2 出現付近のフレーム表示

In [ ]:
# drive_0013 の frame 18, 22, 26, 30 を並べて表示
fig, axes = plt.subplots(4, 1, figsize=(14, 16))
for ax, fidx in zip(axes, [18, 22, 26, 30]):
    img = Image.open(os.path.join(frame_dir_d13, frame_names_d13[fidx]))
    ax.imshow(img)
    ax.set_title(f'drive_0013 - frame {fidx}')
    ax.axis('on')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

###新セル shadow_002-B: state リセット + クリック

In [ ]:
# 既存の inference_state_d13 を再利用（shadow_001 のデータをクリア）
predictor.reset_state(inference_state_d13)

# 車2 を frame 30 でクリック
ann_frame_idx_d13_v2 = 30
click_xy_d13_v2 = (1150, 290)  # 車2 のボディ中央付近、右側

points = np.array([[click_xy_d13_v2[0], click_xy_d13_v2[1]]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)

frame_idx, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state_d13,
    frame_idx=ann_frame_idx_d13_v2,
    obj_id=1,
    points=points,
    labels=labels,
)

mask = (out_mask_logits[0] > 0.0).cpu().numpy()
if mask.ndim == 3:
    mask = mask[0]

img_d13_30 = Image.open(os.path.join(frame_dir_d13, frame_names_d13[ann_frame_idx_d13_v2]))
plt.figure(figsize=(14, 5))
plt.imshow(img_d13_30)
plt.imshow(mask, alpha=0.5, cmap='Reds')
plt.scatter([click_xy_d13_v2[0]], [click_xy_d13_v2[1]], c='lime', s=120, marker='*', edgecolors='black')
plt.title(f'drive_0013 frame {ann_frame_idx_d13_v2} - click {click_xy_d13_v2} + initial mask (car 2)')
plt.axis('off')
plt.show()

###新セル shadow_002-C: 伝播（frame 30 → 143 フォワード）

In [ ]:
video_segments_d13_v2 = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state_d13):
    video_segments_d13_v2[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
print('shadow_002 propagation done. frames covered:', len(video_segments_d13_v2))
print('frame range:', min(video_segments_d13_v2.keys()), '〜', max(video_segments_d13_v2.keys()))

###新セル shadow_002-D: マスク保存（start_frame=30 メタデータ付き）

In [ ]:
CASE_NAME_D13_V2 = 'shadow_002'
START_FRAME_D13_V2 = ann_frame_idx_d13_v2

img0 = Image.open(os.path.join(frame_dir_d13, frame_names_d13[0]))
W, H = img0.size

# frame 30 以降のマスクだけ保存（0〜29 は除外）
total_tracked = len(frame_names_d13) - START_FRAME_D13_V2
masks_arr_d13_v2 = np.zeros((total_tracked, H, W), dtype=bool)
for idx in range(START_FRAME_D13_V2, len(frame_names_d13)):
    if idx in video_segments_d13_v2:
        for oid, m in video_segments_d13_v2[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            masks_arr_d13_v2[idx - START_FRAME_D13_V2] = mask_arr.astype(bool)

save_dir = f'/content/output/{CASE_NAME_D13_V2}'
os.makedirs(save_dir, exist_ok=True)
out_path = f'{save_dir}/masks.npz'
np.savez_compressed(out_path,
                    masks=masks_arr_d13_v2,
                    click_xy=np.array(click_xy_d13_v2, dtype=np.float32),
                    case_name=np.array([CASE_NAME_D13_V2]),
                    H=np.array([H]),
                    W=np.array([W]),
                    fps=np.array([10]),
                    start_frame=np.array([START_FRAME_D13_V2]),
                    source_drive=np.array(['2011_09_26_drive_0013']),
                    condition_tag=np.array(['shadow_alternating + occluder_perspective']))

print(f'masks shape: {masks_arr_d13_v2.shape}')
print(f'non-zero frames: {(masks_arr_d13_v2.sum(axis=(1,2)) > 0).sum()} / {total_tracked}')
print(f'saved to: {out_path}')
print(f'note: indices in masks_arr correspond to original frame indices {START_FRAME_D13_V2} 〜 {len(frame_names_d13)-1}')

###新セル shadow_002-E: オーバーレイ MP4

In [ ]:
import cv2
OVERLAY_DIR_D13_V2 = f'/content/output/{CASE_NAME_D13_V2}/overlay_frames'
os.makedirs(OVERLAY_DIR_D13_V2, exist_ok=True)

# 開始フレーム以降のみ
for idx in range(START_FRAME_D13_V2, len(frame_names_d13)):
    fname = frame_names_d13[idx]
    img_bgr = cv2.imread(os.path.join(frame_dir_d13, fname))
    if idx in video_segments_d13_v2:
        for oid, m in video_segments_d13_v2[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            color = np.array([0, 0, 255], dtype=np.uint8)
            overlay = img_bgr.copy()
            overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 + color * 0.5).astype(np.uint8)
            img_bgr = overlay
    out_idx = idx - START_FRAME_D13_V2
    cv2.imwrite(os.path.join(OVERLAY_DIR_D13_V2, f'{out_idx:05d}.jpg'), img_bgr)

out_mp4 = f'/content/output/{CASE_NAME_D13_V2}/{CASE_NAME_D13_V2}.mp4'
!ffmpeg -y -framerate 10 -i "{OVERLAY_DIR_D13_V2}/%05d.jpg" -c:v libx264 -pix_fmt yuv420p "{out_mp4}" -loglevel error
print('saved:', out_mp4)

In [ ]:
# Colab 上で確認
from IPython.display import Video
Video(out_mp4, embed=True, width=720)

In [ ]:
# zip 圧縮
!zip -r /content/output/{CASE_NAME_D13_V2}.zip /content/output/{CASE_NAME_D13_V2}

# ダウンロード
from google.colab import files
files.download(f"/content/output/{CASE_NAME_D13_V2}.zip")

##drive_0017 (B: 遮蔽)
###新セル 8: drive_0017 フレーム展開

In [ ]:
VIDEO_PATH_D17 = '/content/kitti_drive_0017.mp4'
frame_dir_d17 = '/content/frames_d17'
os.makedirs(frame_dir_d17, exist_ok=True)
!rm -f /content/frames_d17/*.jpg
!ffmpeg -y -i {VIDEO_PATH_D17} -q:v 2 -start_number 0 {frame_dir_d17}/%05d.jpg -loglevel error

frame_names_d17 = sorted([f for f in os.listdir(frame_dir_d17) if f.lower().endswith(('.jpg', '.jpeg'))])
print(f'drive_0017 frames extracted: {len(frame_names_d17)}')

###新セル 9: drive_0017 用 inference_state 初期化

In [ ]:
try:
    predictor.reset_state(inference_state_d17)
except (NameError, AttributeError):
    pass
inference_state_d17 = predictor.init_state(video_path=frame_dir_d17)
print('drive_0017 state initialized')

###新セル 10: 複数フレームを並べて表示（追跡対象を選ぶため）

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16))
for ax, fidx in zip(axes, [0, 20, 35, 50]):
    img = Image.open(os.path.join(frame_dir_d17, frame_names_d17[fidx]))
    ax.imshow(img)
    ax.set_title(f'drive_0017 - frame {fidx}')
    ax.axis('on')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

###新セル 11: クリック + 初期マスク確認

In [ ]:
click_xy_d17 = (10, 220)  # frame 0 のライトイエロー車

points = np.array([[click_xy_d17[0], click_xy_d17[1]]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)

frame_idx, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state_d17,
    frame_idx=0,
    obj_id=1,
    points=points,
    labels=labels,
)

mask = (out_mask_logits[0] > 0.0).cpu().numpy()
if mask.ndim == 3:
    mask = mask[0]

img_d17 = Image.open(os.path.join(frame_dir_d17, frame_names_d17[0]))
plt.figure(figsize=(14, 5))
plt.imshow(img_d17)
plt.imshow(mask, alpha=0.5, cmap='Reds')
plt.scatter([click_xy_d17[0]], [click_xy_d17[1]], c='lime', s=120, marker='*', edgecolors='black')
plt.title(f'drive_0017 frame 0 - click {click_xy_d17} + initial mask')
plt.axis('off')
plt.show()

###新セル 12: 全フレーム伝播

In [ ]:
video_segments_d17 = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state_d17):
    video_segments_d17[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
print('drive_0017 propagation done. frames covered:', len(video_segments_d17))

###新セル 13: マスク保存

In [ ]:
CASE_NAME_D17 = 'pole_occlusion_001'

img0 = Image.open(os.path.join(frame_dir_d17, frame_names_d17[0]))
W, H = img0.size
masks_arr_d17 = np.zeros((len(frame_names_d17), H, W), dtype=bool)
for idx in range(len(frame_names_d17)):
    if idx in video_segments_d17:
        for oid, m in video_segments_d17[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            masks_arr_d17[idx] = mask_arr.astype(bool)

save_dir = f'/content/output/{CASE_NAME_D17}'
os.makedirs(save_dir, exist_ok=True)
out_path = f'{save_dir}/masks.npz'
np.savez_compressed(out_path,
                    masks=masks_arr_d17,
                    click_xy=np.array(click_xy_d17, dtype=np.float32),
                    case_name=np.array([CASE_NAME_D17]),
                    H=np.array([H]),
                    W=np.array([W]),
                    fps=np.array([10]),
                    source_drive=np.array(['2011_09_26_drive_0017']),
                    condition_tag=np.array(['static_camera + pole_occlusion']))

print(f'masks shape: {masks_arr_d17.shape}')
print(f'non-zero frames: {(masks_arr_d17.sum(axis=(1,2)) > 0).sum()} / {len(frame_names_d17)}')
print(f'saved to: {out_path}')
print(f'compressed file size: {os.path.getsize(out_path)/1e6:.2f} MB')

###新セル 14: オーバーレイ MP4

In [ ]:
import cv2
OVERLAY_DIR_D17 = f'/content/output/{CASE_NAME_D17}/overlay_frames'
os.makedirs(OVERLAY_DIR_D17, exist_ok=True)

for idx, fname in enumerate(frame_names_d17):
    img_bgr = cv2.imread(os.path.join(frame_dir_d17, fname))
    if idx in video_segments_d17:
        for oid, m in video_segments_d17[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            color = np.array([0, 0, 255], dtype=np.uint8)
            overlay = img_bgr.copy()
            overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 + color * 0.5).astype(np.uint8)
            img_bgr = overlay
    cv2.imwrite(os.path.join(OVERLAY_DIR_D17, f'{idx:05d}.jpg'), img_bgr)

out_mp4 = f'/content/output/{CASE_NAME_D17}/{CASE_NAME_D17}.mp4'
!ffmpeg -y -framerate 10 -i "{OVERLAY_DIR_D17}/%05d.jpg" -c:v libx264 -pix_fmt yuv420p "{out_mp4}" -loglevel error
print('saved:', out_mp4)

In [ ]:
# Colab 上で確認
from IPython.display import Video
Video(out_mp4, embed=True, width=720)

In [ ]:
!ls -lh /content/output/{CASE_NAME_D17}/

In [ ]:
# zip 圧縮
!zip -r /content/output/{CASE_NAME_D17}.zip /content/output/{CASE_NAME_D17}

# ダウンロード
from google.colab import files
files.download(f"/content/output/{CASE_NAME_D17}.zip")

##Colab で指標計算を全6ケースに実行

In [ ]:
# === Week 2 指標計算 & 可視化（指標4: 形状IoU を追加した更新版）===
import os, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

# --- 指標関数群 ---

def mask_area_change_rate(masks):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    rate = np.zeros(len(areas), dtype=np.float32)
    for t in range(1, len(areas)):
        prev, curr = areas[t-1], areas[t]
        if prev == 0:
            rate[t] = 0.0 if curr == 0 else float('inf')
        else:
            rate[t] = (curr - prev) / prev
    return rate

def connected_components_count(masks, min_size=50):
    counts = np.zeros(len(masks), dtype=np.int32)
    for t in range(len(masks)):
        if masks[t].sum() == 0:
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0: continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        counts[t] = int((sizes >= min_size).sum())
    return counts

def largest_component_ratio(masks):
    ratios = np.ones(len(masks), dtype=np.float32)
    for t in range(len(masks)):
        total = masks[t].sum()
        if total == 0: continue
        labeled, n = ndimage.label(masks[t])
        if n == 0: continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        ratios[t] = float(sizes.max() / total)
    return ratios

def mask_disappearance(masks, threshold_abs=50, threshold_rel=0.01, ref_frames=5):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    ref = areas[:max(ref_frames, 1)]
    ref_area = float(ref.mean()) if len(ref) > 0 else 1.0
    threshold = max(float(threshold_abs), ref_area * threshold_rel)
    is_dis = areas < threshold
    cons = np.zeros(len(areas), dtype=np.int32)
    for t in range(len(areas)):
        cons[t] = (cons[t-1] if t > 0 else 0) + 1 if is_dis[t] else 0
    return {'areas': areas, 'is_disappeared': is_dis, 'consecutive': cons,
            'threshold': threshold, 'reference_area': ref_area}

def shape_iou_temporal(masks):
    """重心揃え後の連続フレーム IoU。位置の変化を打ち消して形状の変化だけを評価"""
    T = len(masks)
    iou = np.ones(T, dtype=np.float32)
    if T == 0: return iou
    H, W = masks[0].shape
    def centroid(m):
        ys, xs = np.where(m)
        return (float(ys.mean()), float(xs.mean())) if len(ys) > 0 else None
    for t in range(1, T):
        mp, mc = masks[t-1], masks[t]
        if mp.sum() == 0 or mc.sum() == 0:
            iou[t] = 1.0
            continue
        cp, cc = centroid(mp), centroid(mc)
        dy = int(round(cp[0] - cc[0]))
        dx = int(round(cp[1] - cc[1]))
        shifted = np.zeros_like(mc)
        y0, y1 = max(0, dy), min(H, H + dy)
        x0, x1 = max(0, dx), min(W, W + dx)
        if y0 < y1 and x0 < x1:
            shifted[y0:y1, x0:x1] = mc[y0-dy:y1-dy, x0-dx:x1-dx]
        inter = np.logical_and(mp, shifted).sum()
        union = np.logical_or(mp, shifted).sum()
        iou[t] = float(inter / union) if union > 0 else 1.0
    return iou

def compute_all_metrics(masks, min_component_size=50):
    area_change = mask_area_change_rate(masks)
    components = connected_components_count(masks, min_size=min_component_size)
    largest = largest_component_ratio(masks)
    dis = mask_disappearance(masks)
    siou = shape_iou_temporal(masks)
    return {
        'frame_count': int(len(masks)),
        'areas': dis['areas'].astype(float).tolist(),
        'area_change_rate': area_change.tolist(),
        'connected_components': components.tolist(),
        'largest_component_ratio': largest.tolist(),
        'shape_iou_temporal': siou.tolist(),
        'is_disappeared': dis['is_disappeared'].tolist(),
        'consecutive_disappearance': dis['consecutive'].tolist(),
        'reference_area': float(dis['reference_area']),
        'disappearance_threshold': float(dis['threshold']),
        'min_component_size': int(min_component_size),
    }

# --- 解析と可視化（5パネル版）---

plt.rcParams['font.family'] = 'DejaVu Sans'

def analyze_one(npz_path, output_dir):
    data = np.load(npz_path, allow_pickle=True)
    masks = data['masks']
    case_name = (str(data['case_name'].item() if 'case_name' in data and data['case_name'].ndim == 0
                     else data['case_name'][0]) if 'case_name' in data
                 else Path(npz_path).parent.name)
    click_xy = data['click_xy'].tolist() if 'click_xy' in data else None
    print(f'[{case_name}] shape={masks.shape}')

    metrics = compute_all_metrics(masks)
    metrics['case_name'] = case_name
    metrics['click_xy'] = click_xy

    os.makedirs(output_dir, exist_ok=True)
    with open(f'{output_dir}/metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    fig, ax = plt.subplots(5, 1, figsize=(12, 13), sharex=True)
    frames = np.arange(len(masks))

    ax[0].plot(frames, metrics['areas'], color='steelblue', linewidth=1.5)
    ax[0].axhline(metrics['disappearance_threshold'], color='red', linestyle='--',
                  label=f"Disappearance threshold ({metrics['disappearance_threshold']:.0f}px)")
    ax[0].set_ylabel('Mask area (px)')
    ax[0].set_title(f'{case_name} - mask area over time')
    ax[0].legend(); ax[0].grid(alpha=0.3)

    rates = np.array(metrics['area_change_rate'])
    ax[1].plot(frames, np.clip(rates, -2, 2), color='darkorange', linewidth=1.5)
    ax[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='±50%')
    ax[1].axhline(-0.5, color='red', linestyle='--', alpha=0.5)
    ax[1].set_ylabel('Area change rate')
    ax[1].set_title('Indicator 1: Mask area change rate')
    ax[1].legend(); ax[1].grid(alpha=0.3)

    ax[2].plot(frames, metrics['connected_components'], color='seagreen',
               linewidth=1.5, marker='o', markersize=3)
    ax[2].set_ylabel('Component count')
    ax[2].set_title(f"Indicator 2: Connected components (>= {metrics['min_component_size']}px)")
    ax[2].grid(alpha=0.3)

    siou = np.array(metrics['shape_iou_temporal'])
    ax[3].plot(frames, siou, color='purple', linewidth=1.5)
    ax[3].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='IoU 0.8')
    ax[3].set_ylim(0, 1.05)
    ax[3].set_ylabel('Shape IoU')
    ax[3].set_title('Indicator 4: Shape stability (centroid-aligned IoU vs previous frame)')
    ax[3].legend(); ax[3].grid(alpha=0.3)

    is_dis = np.array(metrics['is_disappeared'])
    cons = np.array(metrics['consecutive_disappearance'])
    ax[4].fill_between(frames, 0, is_dis.astype(int), alpha=0.4, color='crimson',
                       step='post', label='Disappeared')
    if cons.max() > 0:
        ax[4].plot(frames, cons / max(cons.max(), 1), color='crimson', linewidth=1.5,
                   label='Consecutive (normalized)')
    ax[4].set_ylim(-0.05, 1.1)
    ax[4].set_ylabel('Disappearance')
    ax[4].set_title(f"Indicator 3: Mask disappearance (max consecutive: {int(cons.max())} frames)")
    ax[4].set_xlabel('Frame')
    ax[4].legend(); ax[4].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/metrics.png', dpi=120, bbox_inches='tight')
    plt.close()

    print(f'  -> {output_dir}/metrics.json, metrics.png')
    print(f'  - frames with shape IoU < 0.8: {int((siou < 0.8).sum())}')
    print(f'  - min shape IoU: {float(siou.min()):.3f} at frame {int(siou.argmin())}')
    print(f'  - mean shape IoU (frames 1+): {float(siou[1:].mean()):.3f}')
    print(f'  - disappeared frames: {int(is_dis.sum())} / {len(masks)}')
    print(f'  - max components: {int(max(metrics["connected_components"]))}')
    print(f'  - abrupt area change frames (|rate|>0.5): {int((np.abs(rates) > 0.5).sum())}')
    return metrics

## 全６ケースに実行
cases = ['parked_car_001', 'tram_001', 'cyclist_001', 'shadow_001', 'shadow_002', 'pole_occlusion_001']
all_metrics = {}
for case in cases:
    print('=' * 60)
    npz_path = f'/content/output/{case}/masks.npz'
    if not os.path.exists(npz_path):
        print(f'SKIP: {npz_path} not found')
        continue
    all_metrics[case] = analyze_one(npz_path, f'/content/output/{case}')
print()
print('=' * 60)
print(f'Done. {len(all_metrics)} cases analyzed.')

### zip ダウンロード

In [ ]:
!cd /content/output && \
  zip -r week2_6cases.zip \
    parked_car_001/masks.npz parked_car_001/metrics.json parked_car_001/metrics.png \
    tram_001/masks.npz tram_001/metrics.json tram_001/metrics.png \
    cyclist_001/masks.npz cyclist_001/metrics.json cyclist_001/metrics.png \
    shadow_001/masks.npz shadow_001/metrics.json shadow_001/metrics.png \
    shadow_002/masks.npz shadow_002/metrics.json shadow_002/metrics.png \
    pole_occlusion_001/masks.npz pole_occlusion_001/metrics.json pole_occlusion_001/metrics.png && \
  ls -lh week2_6cases.zip

##week3
###Gradio インストール


In [ ]:
!pip install gradio -q
import gradio as gr
print('gradio:', gr.__version__)

gradio: 5.50.0


In [ ]:
import gradio as gr

def greet(name):
    return f"Hello, {name}! Gradio is working."

demo = gr.Interface(fn=greet, inputs="text", outputs="text", title="SAM2 UI - Hello World")
demo.launch(share=True)

###動画アップロード + フレーム表示 + クリック取得

In [ ]:
import os, uuid, subprocess
from PIL import Image
import gradio as gr

def upload_video(video_path, state):
    """動画をアップロード、ffmpeg でフレーム展開、最初のフレームを返す"""
    if video_path is None:
        return None, state, "動画が選択されていません"

    # 一意なフレームディレクトリを作成
    frame_dir = f'/tmp/gradio_frames_{uuid.uuid4().hex[:8]}'
    os.makedirs(frame_dir, exist_ok=True)

    # ffmpeg でフレーム展開
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-q:v', '2', '-start_number', '0',
        f'{frame_dir}/%05d.jpg',
        '-loglevel', 'error',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None, state, f"ffmpeg error: {result.stderr[:200]}"

    frame_names = sorted([f for f in os.listdir(frame_dir) if f.lower().endswith('.jpg')])
    if not frame_names:
        return None, state, "フレームが展開できませんでした"

    state['frame_dir'] = frame_dir
    state['frame_names'] = frame_names
    state['click_xy'] = None

    first_frame = Image.open(os.path.join(frame_dir, frame_names[0]))
    msg = f"OK: {len(frame_names)} frames extracted. Size: {first_frame.size}"
    return first_frame, state, msg


def on_click(state, evt: gr.SelectData):
    """画像クリック座標を取得"""
    if state.get('frame_names') is None:
        return state, "先に動画をアップロードしてください"
    x, y = evt.index[0], evt.index[1]
    state['click_xy'] = (x, y)
    return state, f"クリック座標: ({x}, {y})"


with gr.Blocks(title="SAM2 UI - Week 3-A") as demo:
    state = gr.State({})

    gr.Markdown("# SAM2 認識限界観測システム — Prototype")
    gr.Markdown("**Step 1**: 動画をアップロード → **Step 2**: 表示されたフレームをクリックで対象指定")

    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="動画ファイル (mp4)")
            upload_btn = gr.Button("アップロードしてフレーム展開", variant="primary")
            upload_status = gr.Textbox(label="ステータス", interactive=False)
        with gr.Column():
            frame_display = gr.Image(label="最初のフレーム（クリックで対象指定）")
            click_status = gr.Textbox(label="クリック座標", interactive=False)

    upload_btn.click(
        upload_video,
        inputs=[video_input, state],
        outputs=[frame_display, state, upload_status],
    )
    frame_display.select(
        on_click,
        inputs=[state],
        outputs=[state, click_status],
    )

demo.launch(share=True, debug=False)

###「Run Tracking」ボタンを押すと SAM2 が起動して tracked.mp4 を出力

In [6]:
import os, uuid, subprocess, shutil
import numpy as np
import cv2
from PIL import Image
import gradio as gr

# === 前提確認 ===
assert 'predictor' in dir(), "先に SAM2 predictor をロードしてください（既存ノートブックのセル 5）"


def upload_video(video_path, state):
    """動画アップロード + フレーム展開 + SAM2 inference_state 初期化"""
    if video_path is None:
        return None, state, "動画が選択されていません"

    # 前回のフレームディレクトリがあれば削除
    old_dir = state.get('frame_dir')
    if old_dir and os.path.exists(old_dir):
        shutil.rmtree(old_dir, ignore_errors=True)

    frame_dir = f'/tmp/gradio_frames_{uuid.uuid4().hex[:8]}'
    os.makedirs(frame_dir, exist_ok=True)

    cmd = ['ffmpeg', '-y', '-i', video_path, '-q:v', '2',
           '-start_number', '0', f'{frame_dir}/%05d.jpg', '-loglevel', 'error']
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None, state, f"ffmpeg error: {result.stderr[:200]}"

    frame_names = sorted([f for f in os.listdir(frame_dir) if f.lower().endswith('.jpg')])
    if not frame_names:
        return None, state, "フレームが展開できませんでした"

    # SAM2 inference_state を初期化（フレーム数次第で 10〜30秒）
    inference_state = predictor.init_state(video_path=frame_dir)

    state['frame_dir'] = frame_dir
    state['frame_names'] = frame_names
    state['inference_state'] = inference_state
    state['click_xy'] = None
    state['video_segments'] = None

    first_frame = Image.open(os.path.join(frame_dir, frame_names[0]))
    msg = f"OK: {len(frame_names)} frames extracted. Size: {first_frame.size}. SAM2 state initialized."
    return first_frame, state, msg


def on_click(state, evt: gr.SelectData):
    if state.get('frame_names') is None:
        return state, "先に動画をアップロードしてください"
    x, y = evt.index[0], evt.index[1]
    state['click_xy'] = (x, y)
    return state, f"クリック座標: ({x}, {y})"


def run_tracking(state, progress=gr.Progress()):
    """SAM2 追跡を実行、tracked.mp4 を返す"""
    if state.get('click_xy') is None:
        return None, state, "先にフレームをクリックして対象を指定してください"
    if state.get('inference_state') is None:
        return None, state, "先に動画をアップロードしてください"

    inference_state = state['inference_state']

    progress(0.05, desc="State リセット")
    predictor.reset_state(inference_state)

    progress(0.15, desc="クリック点追加")
    click_xy = state['click_xy']
    points = np.array([click_xy], dtype=np.float32)
    labels = np.array([1], dtype=np.int32)
    predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0, obj_id=1, points=points, labels=labels,
    )

    progress(0.25, desc="全フレーム伝播中...")
    video_segments = {}
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        video_segments[out_frame_idx] = {
            oid: (out_mask_logits[i] > 0.0).cpu().numpy()
            for i, oid in enumerate(out_obj_ids)
        }
    state['video_segments'] = video_segments

    progress(0.75, desc="オーバーレイ生成中...")
    frame_dir = state['frame_dir']
    frame_names = state['frame_names']
    out_dir = f'/tmp/gradio_out_{uuid.uuid4().hex[:8]}'
    overlay_dir = os.path.join(out_dir, 'overlay')
    os.makedirs(overlay_dir, exist_ok=True)

    for idx, fname in enumerate(frame_names):
        img_bgr = cv2.imread(os.path.join(frame_dir, fname))
        if idx in video_segments:
            for oid, m in video_segments[idx].items():
                mask_arr = m
                if mask_arr.ndim == 3:
                    mask_arr = mask_arr[0]
                color = np.array([0, 0, 255], dtype=np.uint8)
                overlay = img_bgr.copy()
                overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 + color * 0.5).astype(np.uint8)
                img_bgr = overlay
        cv2.imwrite(os.path.join(overlay_dir, f'{idx:05d}.jpg'), img_bgr)

    progress(0.9, desc="MP4 化中...")
    out_mp4 = os.path.join(out_dir, 'tracked.mp4')
    cmd = ['ffmpeg', '-y', '-framerate', '10', '-i', f'{overlay_dir}/%05d.jpg',
           '-c:v', 'libx264', '-pix_fmt', 'yuv420p', out_mp4, '-loglevel', 'error']
    subprocess.run(cmd, capture_output=True, text=True)

    state['tracked_mp4'] = out_mp4

    # サマリ
    total = len(frame_names)
    tracked = sum(1 for f in video_segments.values()
                  if any(m.sum() > 50 if m.ndim == 2 else m[0].sum() > 50 for m in f.values()))
    summary = (f"完了: {total} フレーム中 {tracked} フレームでマスクあり\n"
               f"クリック: {click_xy}\n"
               f"出力: {out_mp4}")

    progress(1.0, desc="完了")
    return out_mp4, state, summary


with gr.Blocks(title="SAM2 UI - Week 3-A") as demo:
    state = gr.State({})

    gr.Markdown("# SAM2 認識限界観測システム — Prototype (Week 3-A)")

    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="動画ファイル (mp4)")
            upload_btn = gr.Button("アップロードして展開", variant="primary")
            upload_status = gr.Textbox(label="アップロードステータス", interactive=False)
        with gr.Column():
            frame_display = gr.Image(label="最初のフレーム（クリックで対象指定）")
            click_status = gr.Textbox(label="クリック座標", interactive=False)

    with gr.Row():
        run_btn = gr.Button("Run Tracking (SAM2 伝播)", variant="primary", size="lg")

    with gr.Row():
        with gr.Column():
            tracked_video = gr.Video(label="追跡結果")
        with gr.Column():
            tracking_summary = gr.Textbox(label="サマリ", interactive=False, lines=5)

    upload_btn.click(upload_video, inputs=[video_input, state],
                     outputs=[frame_display, state, upload_status])
    frame_display.select(on_click, inputs=[state],
                          outputs=[state, click_status])
    run_btn.click(run_tracking, inputs=[state],
                  outputs=[tracked_video, state, tracking_summary])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b993c834309bb9256.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
